In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
!pip install transformers accelerate peft bitsandbytes
!pip install llama-cpp-python
!pip install pandas psutil

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.7/50.7 MB 25.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.7 MB/s eta 0:00:00
  Created wheel for llama-cpp-python: filename=llama_cpp_python-0.3.16-cp312-cp312-linux_x86_64.whl size=4422335 sha256=18c7a6915a7819f206a7ccd6daa3f24557bf35fe0b416bd6ecf1f7af809d5a03
  Stored in directory: /root/.cache/pip/wheels/90/82/ab/8784ee3fb99ddb07fd36a679ddbe63122cc07718f6c1eb3be8
Successfully built llama-cpp-python


In [10]:
BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

ADAPTER_PATH = "/content/drive/MyDrive/WEEK-8/adapters"

GGUF_MODEL = "/content/drive/MyDrive/WEEK-8/quantized_models/model-q8_0.gguf"

In [5]:
import torch
import time
import pandas as pd
import psutil

from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

In [6]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    device_map="auto",
    torch_dtype=torch.float16
)

base_model.eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear(in_features=1536, out_features=1536, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((1536,), eps=1e-06)
    (rotar

In [8]:
finetuned_model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_PATH
)

finetuned_model.eval()

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 1536)
        (layers): ModuleList(
          (0-27): 28 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=1536, out_features=1536, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1536, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=1536, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): Linear(in_fe

In [11]:
from llama_cpp import Llama

gguf_model = Llama(
    model_path=GGUF_MODEL,
    n_ctx=2048
)

llama_model_loader: loaded meta data with 26 key-value pairs and 338 tensors from /content/drive/MyDrive/WEEK-8/quantized_models/model-q8_0.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = qwen2
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                     general.sampling.top_k i32              = 20
llama_model_loader: - kv   3:                     general.sampling.top_p f32              = 0.800000
llama_model_loader: - kv   4:                      general.sampling.temp f32              = 0.700000
llama_model_loader: - kv   5:                               general.name str              = Merged_Model
llama_model_loader: - kv   6:                         general.size_label str              = 1.5B
llama_model_loader: - kv   7:      

In [12]:
prompts = [
    "What is bipolar disorder?",
    "Why should patients with suspected major depressive disorder be screened for manic episodes?",
    "Explain peripheral arterial disease treatment.",
    "What is zero order drug elimination?",
    "What hormonal imbalance occurs in polycystic ovarian syndrome?"
]

In [14]:
import time
import torch

def benchmark_transformer(model, tokenizer, prompts):

    total_latency = 0
    total_tokens = 0

    torch.cuda.empty_cache()

    start_vram = torch.cuda.memory_allocated() if torch.cuda.is_available() else 0

    for prompt in prompts:

        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

        start = time.time()

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=120
            )

        end = time.time()

        latency = end - start

        generated_tokens = outputs.shape[-1] - inputs["input_ids"].shape[-1]

        total_latency += latency
        total_tokens += generated_tokens

    end_vram = torch.cuda.memory_allocated() if torch.cuda.is_available() else 0

    avg_latency = total_latency / len(prompts)
    tokens_per_sec = total_tokens / total_latency
    vram_used = (end_vram - start_vram) / (1024**2)

    return avg_latency, tokens_per_sec, vram_used

In [15]:
#base model benchmark
base_latency, base_tps, base_vram = benchmark_transformer(
    base_model,
    tokenizer,
    prompts
)

In [16]:
finetune_latency, finetune_tps, finetune_vram = benchmark_transformer(
    finetuned_model,
    tokenizer,
    prompts
)

In [17]:
def benchmark_gguf(model, prompts):

    total_latency = 0
    total_tokens = 0

    for prompt in prompts:

        start = time.time()

        output = model(
            prompt,
            max_tokens=120
        )

        end = time.time()

        latency = end - start
        tokens = output["usage"]["completion_tokens"]

        total_latency += latency
        total_tokens += tokens

    avg_latency = total_latency / len(prompts)
    tokens_per_sec = total_tokens / total_latency

    return avg_latency, tokens_per_sec


gguf_latency, gguf_tps = benchmark_gguf(gguf_model, prompts)
gguf_vram = 0  # CPU inference

llama_perf_context_print:        load time =    4662.43 ms
llama_perf_context_print: prompt eval time =    4661.57 ms /     5 tokens (  932.31 ms per token,     1.07 tokens per second)
llama_perf_context_print:        eval time =   32963.48 ms /   119 runs   (  277.00 ms per token,     3.61 tokens per second)
llama_perf_context_print:       total time =   37895.68 ms /   124 tokens
llama_perf_context_print:    graphs reused =        115
llama_perf_context_print:        load time =    4662.43 ms
llama_perf_context_print: prompt eval time =     932.38 ms /    14 tokens (   66.60 ms per token,    15.02 tokens per second)
llama_perf_context_print:        eval time =   28617.81 ms /   119 runs   (  240.49 ms per token,     4.16 tokens per second)
llama_perf_context_print:       total time =   29788.71 ms /   133 tokens
llama_perf_context_print:    graphs reused =        114
llama_perf_context_print:        load time =    4662.43 ms
llama_perf_context_print: prompt eval time =     607.23 ms 

In [18]:
test_set = [
    ("What is bipolar disorder?", "manic"),
    ("What drug therapy reduces PAD mortality?", "statin"),
    ("What staining detects iron in tissues?", "prussian"),
]

def evaluate_accuracy(model, tokenizer, test_set):

    correct = 0

    for q, keyword in test_set:

        inputs = tokenizer(q, return_tensors="pt").to(model.device)

        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=80)

        text = tokenizer.decode(outputs[0], skip_special_tokens=True).lower()

        if keyword in text:
            correct += 1

    return correct / len(test_set)

In [19]:
base_acc = evaluate_accuracy(base_model, tokenizer, test_set)
finetune_acc = evaluate_accuracy(finetuned_model, tokenizer, test_set)

In [20]:
def evaluate_accuracy_gguf(model, test_set):

    correct = 0

    for q, keyword in test_set:

        out = model(q, max_tokens=80)

        text = out["choices"][0]["text"].lower()

        if keyword in text:
            correct += 1

    return correct / len(test_set)

gguf_acc = evaluate_accuracy_gguf(gguf_model, test_set)

Llama.generate: 1 prefix-match hit, remaining 4 prompt tokens to eval
llama_perf_context_print:        load time =    4662.43 ms
llama_perf_context_print: prompt eval time =    2916.44 ms /     4 tokens (  729.11 ms per token,     1.37 tokens per second)
llama_perf_context_print:        eval time =   18963.20 ms /    79 runs   (  240.04 ms per token,     4.17 tokens per second)
llama_perf_context_print:       total time =   22035.14 ms /    83 tokens
llama_perf_context_print:    graphs reused =         76
Llama.generate: 1 prefix-match hit, remaining 6 prompt tokens to eval
llama_perf_context_print:        load time =    4662.43 ms
llama_perf_context_print: prompt eval time =     479.77 ms /     6 tokens (   79.96 ms per token,    12.51 tokens per second)
llama_perf_context_print:        eval time =   19591.39 ms /    79 runs   (  247.99 ms per token,     4.03 tokens per second)
llama_perf_context_print:       total time =   20224.78 ms /    85 tokens
llama_perf_context_print:    graph

In [21]:
batch_inputs = tokenizer(prompts, return_tensors="pt", padding=True).to(base_model.device)

with torch.no_grad():
    batch_outputs = base_model.generate(
        **batch_inputs,
        max_new_tokens=80
    )

batch_results = tokenizer.batch_decode(batch_outputs, skip_special_tokens=True)

batch_results

A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.


['What is bipolar disorder? Bipolar Disorder, also known as manic-depressive illness or mania, is a mental health condition that causes dramatic changes in mood and energy levels. People with bipolar disorder experience periods of depression when they feel sad, hopeless, and worthless; and periods of mania when they are extremely excited, irritable, and easily distracted.\n\nBipolar disorder can be difficult to diagnose because the symptoms can overlap',
 'Why should patients with suspected major depressive disorder be screened for manic episodes? Which of the following statements is incorrect?\nA. Patients who have had a history of mania, especially recurrent episodes.\nB. Patients with significant family history of bipolar disorder or schizophrenia.\nC. Individuals under 25 years old.\nD. Patients who have previously been diagnosed with depression but are now experiencing new symptoms that may indicate mania.\nE. Patients with a first-degree relative (parent,',
 'Explain peripheral a

In [22]:
for chunk in gguf_model(
    "Explain peripheral arterial disease.",
    max_tokens=100,
    stream=True
):
    print(chunk["choices"][0]["text"], end="", flush=True)

 Peripheral arterial disease is a condition that affects the blood vessels that supply blood to the arms and legs. It occurs when plaque builds up in the walls of the arteries, causing them to narrow and restrict blood flow. This can lead to reduced blood flow to the limbs, which can cause pain, numbness, and other symptoms. Peripheral arterial disease is more common in smokers, people with diabetes, and those who have high blood pressure or high cholesterol levels. It can also be caused by atherosclerosis,

llama_perf_context_print:        load time =    4662.43 ms
llama_perf_context_print: prompt eval time =    2915.98 ms /     6 tokens (  486.00 ms per token,     2.06 tokens per second)
llama_perf_context_print:        eval time =   25597.57 ms /    99 runs   (  258.56 ms per token,     3.87 tokens per second)
llama_perf_context_print:       total time =   28877.74 ms /   105 tokens
llama_perf_context_print:    graphs reused =         95


In [23]:
import pandas as pd

benchmark_table = pd.DataFrame({
    "Model": ["Base", "Fine-tuned", "GGUF"],
    "Latency (s)": [base_latency, finetune_latency, gguf_latency],
    "Tokens/sec": [base_tps, finetune_tps, gguf_tps],
    "VRAM (MB)": [base_vram, finetune_vram, gguf_vram],
    "Accuracy": [base_acc, finetune_acc, gguf_acc],
    "Device": ["CPU", "CPU", "CPU"]
})

benchmark_table

,Model,Latency (s),Tokens/sec,VRAM (MB),Accuracy,Device
0,Base,76.951775,1.559418,0.0,0.666667,CPU
1,Fine-tuned,70.336069,1.706095,0.0,0.333333,CPU
2,GGUF,31.737416,3.781026,0.0,0.666667,CPU


In [24]:
import psutil

ram = psutil.Process().memory_info().rss / (1024**2)
print("RAM usage (MB):", ram)

RAM usage (MB): 5631.19921875
